# All-runs comparison

Same plot types as the nb_steps comparison quick notebook, but for **all completed runs** in `data/simulations/`. Only runs with training completed for all three schedules (same, near, far) are included.

## 1. Setup and paths

In [1]:
import sys
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from a1b2.utils.run_config import build_run_id
from a1b2.analysis import transfer_interference as ann
from a1b2.utils.figure_settings import schedule_colours, condition_order, task_colours, cm_conv
from a1b2.utils.figure_utils import _style_axes, plot_split_stim, get_axis_limits

data_folder = project_root / "data"
sim_folder = data_folder / "simulations"
config_path = project_root / "a1b2" / "models" / "experiments.json"
with open(config_path, "r") as f:
    settings = json.load(f)
print("Project root:", project_root)
print("Simulations folder:", sim_folder)

Project root: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular
Simulations folder: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations


## 2. Discover run folders

Scan `data/simulations/` for run folders (two_module_rnn, single_module_rnn, or nb_steps_comparison subdirs). Build list of (run_id, path, condition_dict).

In [2]:
with open(config_path, "r") as f:
    settings = json.load(f)

rnn_conditions = [c for c in settings["conditions"] if c.get("arch") in ("two_module_rnn", "single_module_rnn")]
expected_run_ids = {build_run_id(c) for c in rnn_conditions}

existing_dirs = []
if sim_folder.exists():
    for d in sim_folder.iterdir():
        if not d.is_dir():
            continue
        if d.name.startswith("two_module_rnn") or d.name.startswith("single_module_rnn") or d.name in expected_run_ids:
            existing_dirs.append(d.name)
    comp = sim_folder / "nb_steps_comparison"
    if comp.exists():
        for d in comp.iterdir():
            if d.is_dir():
                existing_dirs.append(d.name)

runs = []
for run_id in sorted(set(existing_dirs)):
    path = sim_folder / run_id if (sim_folder / run_id).exists() else sim_folder / "nb_steps_comparison" / run_id
    if not path.exists():
        continue
    cond = None
    sp = path / "settings.json"
    if sp.exists():
        with open(sp, "r") as f:
            cond = json.load(f).get("condition")
    runs.append((run_id, path, cond))

print("Candidate run folders:", len(runs))

Candidate run folders: 40


## 3. Load data (completed only)

Include only runs where all three schedules (same, near, far) have at least one participant. Build `run_labels` and a short label helper for titles.

In [ ]:
from tqdm.auto import tqdm

# 100% completed = exactly these participant counts per schedule
EXPECTED_SAME, EXPECTED_NEAR, EXPECTED_FAR = 103, 101, 101

all_ann_data = {}
skipped = []

skip_nb1 = True
skip_nb2 = False
skip_50 = True
skip_25 = False

for run_id, path, _ in tqdm(runs, desc="Loading runs"):
    if not path.exists():
        skipped.append((run_id, "path does not exist"))
        continue
    ann_data = ann.load_ann_data(str(path), load_rnn_extra=True)
    n_same = len(ann_data["same"])
    n_near = len(ann_data["near"])
    n_far = len(ann_data["far"])
    if n_same == 0 or n_near == 0 or n_far == 0:
        skipped.append((run_id, f"incomplete: same={n_same}, near={n_near}, far={n_far}"))
        continue
    if (n_same, n_near, n_far) != (EXPECTED_SAME, EXPECTED_NEAR, EXPECTED_FAR):
        skipped.append((run_id, f"incomplete: same={n_same}, near={n_near}, far={n_far} (expected {EXPECTED_SAME}, {EXPECTED_NEAR}, {EXPECTED_FAR})"))
        continue
    # Skip runs with nb1, so all that have nb1 in their run_id somewhere
    if skip_nb1 and "nb1" in run_id:
        skipped.append((run_id, "skipping nb1 run"))
        continue
    # Skip runs with module size 50
    if skip_50 and "_50_" in run_id:
        skipped.append((run_id, "skipping 50 run"))
        continue
        # Skip runs with module size 25
    if skip_25 and "_25_" in run_id:
        skipped.append((run_id, "skipping 25 run"))
        continue
    # Skip runs with nb2
    if skip_nb2 and "nb2" in run_id:
        skipped.append((run_id, "skipping nb2 run"))
        continue
    all_ann_data[run_id] = ann_data

run_labels = sorted(all_ann_data.keys())
R = len(run_labels)
schedules = condition_order

def short_label(run_id):
    s = run_id.replace("two_module_rnn_50_", "").replace("single_module_rnn_50_", "").replace("two_module_rnn_25_", "").replace("single_module_rnn_25_", "")
    return s[:24] + ("…" if len(s) > 24 else "")

def display_label(run_id):
    """Short readable config: module size (h25/h50), nb1/nb2, sh/tr, sparsity; 1-mod for single module."""
    prefix = "1-mod: " if run_id.startswith("single_module") else ""
    h = "h25" if (run_id.startswith("two_module_rnn_25") or run_id.startswith("single_module_rnn_25") or "_25_" in run_id) else "h50"
    s = run_id.replace("two_module_rnn_50_", "").replace("single_module_rnn_50_", "").replace("two_module_rnn_25_", "").replace("single_module_rnn_25_", "")
    parts = s.split("_")
    nb = next((p for p in parts if p.startswith("nb") and len(p) <= 4), "?")
    routing = "tr" if "task_routed" in run_id else "sh"
    sp = next((p for p in parts if p.startswith("sp")), "")
    if sp == "sp1.0": sp = "sp1"
    rest = (f"{nb} {routing} {sp}" if sp else f"{nb} {routing}")
    return prefix + h + " " + rest

print("Loaded", R, "completed run(s).")
if skipped:
    print("Skipped:", skipped)
for rid in run_labels:
    d = all_ann_data[rid]
    print(" ", display_label(rid), "same=%d near=%d far=%d" % (len(d["same"]), len(d["near"]), len(d["far"])))

/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading runs:   5%|▌         | 2/40 [00:12<04:06,  6.48s/it]

### Run index to configuration

Reference for subplot rows: index (1-based) maps to display label and full run_id.

In [ ]:
for i, rid in enumerate(run_labels):
    print(f"{i+1}: {display_label(rid)}  |  {rid}")


## 4. Compute metrics

Transfer, loss (and Holton-style trained-per-phase), PCA, principal angles. Detect runs with trajectory data for later plots.

In [ ]:
transfer_dfs = []
for run_id, data in all_ann_data.items():
    df = ann.compute_transfer_anns(data)
    df["run_label"] = run_id
    transfer_dfs.append(df)
transfer_all = pd.concat(transfer_dfs, ignore_index=True)

loss_results = {}
loss_results_holton = {}
for run_id, data in all_ann_data.items():
    loss_results[run_id] = ann.analyze_training_loss(data)
    loss_results_holton[run_id] = ann.analyze_training_loss(data, feature_style='trained_per_phase')

pca_dfs = []
for run_id, data in all_ann_data.items():
    df = ann.compute_pca_components(data, variance_threshold=0.99)
    df["run_label"] = run_id
    pca_dfs.append(df)
pca_all = pd.concat(pca_dfs, ignore_index=True)

angle_dfs = []
for run_id, data in all_ann_data.items():
    df = ann.get_principal_angles(data)
    df["run_label"] = run_id
    angle_dfs.append(df)
angles_all = pd.concat(angle_dfs, ignore_index=True)

runs_with_trajectory = []
for run_id in run_labels:
    if not all_ann_data[run_id].get("same"):
        continue
    sample = all_ann_data[run_id]["same"][0]
    if any("trajectory" in str(k) for k in sample.keys()):
        runs_with_trajectory.append(run_id)

print("Transfer sample:")
print(transfer_all.groupby(["run_label", "condition"])["error_diff"].agg(["mean", "sem"]).head(9))

### Layout constants

Subplot size and max rows per figure so we split into multiple figures when R is large.

In [ ]:
SUBPLOT_W = 2.2
SUBPLOT_H = 2.0
MAX_ROWS_PER_FIGURE = 8

def get_figure_chunks(n_total, ncols=3):
    start = 0
    while start < n_total:
        end = min(start + MAX_ROWS_PER_FIGURE, n_total)
        yield start, end
        start = end

## 5. Transfer (Task B Δ accuracy)

One subplot per run; x = same/near/far, y = mean error_diff. Wrapped grid when R > 6.

In [ ]:
if R == 0:
    print("No runs loaded; skip transfer plot.")
else:
    agg = transfer_all.groupby(["run_label", "condition"], as_index=False)["error_diff"].agg(["mean", "sem"]).reset_index()
    ncols = min(R, 6)
    nrows = (R + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(SUBPLOT_W * ncols, SUBPLOT_H * nrows), squeeze=False, sharey=True)
    y_min = transfer_all["error_diff"].min() - 0.05
    y_max = transfer_all["error_diff"].max() + 0.05
    x_t = np.arange(len(condition_order))
    for idx, run_label in enumerate(run_labels):
        ax = axes.flat[idx]
        sub = agg[agg["run_label"] == run_label]
        if sub.empty:
            ax.set_visible(False)
            continue
        means = [sub[(sub["condition"] == s)]["mean"].values for s in condition_order]
        means = [float(m[0]) if len(m) else np.nan for m in means]
        ax.plot(x_t, means, color="k", linewidth=1)
        for c, (s, m) in enumerate(zip(condition_order, means)):
            if not np.isnan(m):
                ax.scatter(c, m, facecolors="white", edgecolors=schedule_colours[c], linewidths=1.5, s=50, zorder=3)
        ax.set_xticks(x_t)
        ax.set_xticklabels(condition_order)
        ax.set_xlim(-0.5, len(condition_order) - 0.5)
        ax.set_ylim(y_min, y_max)
        ax.set_ylabel("Δ accuracy")
        ax.set_title(display_label(run_label))
        _style_axes(ax)
    for idx in range(R, nrows * ncols):
        axes.flat[idx].set_visible(False)
    plt.tight_layout()
    plt.show()

## 6. Retest interference (1 - A2 accuracy, Near/Far only)

In [ ]:
interference_order = ["near", "far"]
if R == 0:
    print("No runs; skip.")
else:
    interference_rows = []
    for run_label in run_labels:
        for sched in interference_order:
            if sched not in all_ann_data[run_label] or len(all_ann_data[run_label][sched]) == 0:
                continue
            for subj in range(len(all_ann_data[run_label][sched])):
                acc = all_ann_data[run_label][sched][subj].get("accuracy")
                if acc is None or acc.ndim < 2:
                    continue
                a2_acc = np.mean(acc[2, 1::2])
                interference_rows.append({"run_label": run_label, "condition": sched, "interference": 1 - a2_acc})
    interference_df = pd.DataFrame(interference_rows)
    agg_int = interference_df.groupby(["run_label", "condition"], as_index=False)["interference"].agg(["mean", "sem"]).reset_index()
    ncols = min(R, 6)
    nrows = (R + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(SUBPLOT_W * ncols, SUBPLOT_H * nrows), squeeze=False, sharey=True)
    for idx, run_label in enumerate(run_labels):
        ax = axes.flat[idx]
        sub = agg_int[agg_int["run_label"] == run_label]
        if sub.empty:
            ax.set_visible(False)
            continue
        means = [sub[(sub["condition"] == s)]["mean"].values for s in interference_order]
        means = [float(m[0]) if len(m) else np.nan for m in means]
        for c, (s, m) in enumerate(zip(interference_order, means)):
            if not np.isnan(m):
                ax.scatter(c, m, facecolors="white", edgecolors=schedule_colours[condition_order.index(s)], linewidths=1.5, s=50, zorder=3)
        ax.set_xticks(range(len(interference_order)))
        ax.set_xticklabels(interference_order)
        ax.set_ylim(-0.1, 1.1)
        ax.set_ylabel("interference")
        ax.set_xlim(-0.5, len(interference_order) - 0.5)
        ax.set_title(display_label(run_label))
        _style_axes(ax)
    for idx in range(R, nrows * ncols):
        axes.flat[idx].set_visible(False)
    plt.tight_layout()
    plt.show()

## 7. Accuracy timecourse (summer & winter, A1 + B)

R×3 subplots; split into multiple figures when R > MAX_ROWS_PER_FIGURE.

In [ ]:
import seaborn as sns

acc_timecourse_rows = []
n_blocks_per_phase = 10
if all_ann_data:
    for run_label in run_labels:
        for sched in schedules:
            if sched not in all_ann_data[run_label] or len(all_ann_data[run_label][sched]) == 0:
                continue
            for subj_idx, subj_data in enumerate(all_ann_data[run_label][sched]):
                acc = subj_data.get("accuracy")
                if acc is None or acc.ndim < 2:
                    continue
                n_phase, n_trials = acc.shape[0], acc.shape[1]
                for phase, task_section in enumerate(("A1", "B")):
                    if phase >= n_phase:
                        break
                    block_size = max(1, n_trials // n_blocks_per_phase)
                    for b in range(n_blocks_per_phase):
                        start = b * block_size
                        end = min((b + 1) * block_size, n_trials)
                        if start >= end:
                            continue
                        global_block = b + phase * n_blocks_per_phase
                        summer_acc = np.nanmean(acc[phase, start:end:2])
                        winter_acc = np.nanmean(acc[phase, start+1:end:2]) if end - start > 1 else np.nan
                        acc_timecourse_rows.append({"run_label": run_label, "condition": sched, "participant": subj_idx, "block": global_block, "task_section": "A1" if phase == 0 else "B", "feature_idx": 0, "feature": "summer accuracy", "accuracy": summer_acc})
                        acc_timecourse_rows.append({"run_label": run_label, "condition": sched, "participant": subj_idx, "block": global_block, "task_section": "A1" if phase == 0 else "B", "feature_idx": 1, "feature": "winter accuracy", "accuracy": winter_acc})
    acc_trial_df = pd.DataFrame(acc_timecourse_rows)
    for start_r, end_r in get_figure_chunks(R, 3):
        nrows_fig = end_r - start_r
        run_slice = run_labels[start_r:end_r]
        fig, axes = plt.subplots(nrows_fig, 3, figsize=(3 * SUBPLOT_W, nrows_fig * SUBPLOT_H), sharex=True, sharey=True)
        if nrows_fig == 1:
            axes = axes.reshape(1, -1)
        for r, run_label in enumerate(run_slice):
            for c, sched in enumerate(schedules):
                ax = axes[r, c]
                sub = acc_trial_df[(acc_trial_df["run_label"] == run_label) & (acc_trial_df["condition"] == sched)]
                if sub.empty:
                    ax.set_visible(False)
                    continue
                sns.lineplot(ax=ax, data=sub, x="block", y="accuracy", hue="feature", errorbar="se", hue_order=["summer accuracy", "winter accuracy"], linewidth=1)
                ax.axvline(10, linestyle="--", color="k", linewidth=0.5)
                ax.axhline(0.5, linestyle="-", color="grey", linewidth=5, alpha=0.2)
                ax.set_ylabel("accuracy")
                ax.set_ylim([0.4, 1])
                ax.set_xticks(range(0, 21, 10))
                ax.set_title(display_label(run_label) + " | " + sched)
                if c == 0:
                    ax.set_ylabel(display_label(run_label), fontsize=7)
                _style_axes(ax)
                if c == 0:
                    ax.legend(loc="center left", fontsize=6)
                else:
                    if ax.get_legend():
                        ax.get_legend().remove()
        plt.tight_layout()
        plt.show()
else:
    print("No data; skip accuracy timecourse.")

## 8. Loss curves (mean loss over epochs)

In [ ]:
# Same variant as steps comparison: winter probe, rolling smooth, red phase boundaries, "Mean loss"
if R == 0:
    print("No runs; skip.")
else:
    smooth_window = 5
    for start_r, end_r in get_figure_chunks(R, 3):
        nrows_fig = end_r - start_r
        run_slice = run_labels[start_r:end_r]
        fig, axes = plt.subplots(nrows_fig, 3, figsize=(3 * SUBPLOT_W, nrows_fig * SUBPLOT_H), sharex=True, sharey=True)
        if nrows_fig == 1:
            axes = axes.reshape(1, -1)
        for r, run_label in enumerate(run_slice):
            for c, sched in enumerate(schedules):
                ax = axes[r, c]
                if run_label not in loss_results or sched not in loss_results[run_label]:
                    ax.set_visible(False)
                    continue
                loss = loss_results[run_label][sched]
                mean_loss = np.asarray(loss["mean"])
                std_loss = loss.get("std", np.zeros_like(mean_loss))
                steps = np.arange(len(mean_loss))
                smooth_mean = pd.Series(mean_loss).rolling(window=smooth_window, center=True, min_periods=1).mean().to_numpy()
                color = schedule_colours[c]
                if np.any(np.isfinite(std_loss)) and np.any(std_loss > 0):
                    ax.fill_between(steps, mean_loss - std_loss, mean_loss + std_loss, color=color, alpha=0.25)
                ax.plot(steps, smooth_mean, color=color, linewidth=1.5, alpha=0.9)
                bounds = loss.get("phase_boundaries")
                if bounds is not None:
                    for x in bounds:
                        ax.axvline(x, color="red", linestyle="--", linewidth=1)
                ax.set_xlabel("Batch (training step)")
                ax.set_title(display_label(run_label) + " | " + sched)
                if c == 0:
                    ax.set_ylabel(display_label(run_label), fontsize=7)
                else:
                    ax.set_ylabel("Mean loss")
                _style_axes(ax)
        fig.suptitle("Training loss (winter probe, training only). A2: winter untrained.", y=1.02)
        plt.tight_layout()
        plt.show()

## 9. Loss + Holton (trained per phase) overlay

In [ ]:
# Same variant as steps comparison: Holton style — solid winter (A2 untrained), dashed trained per phase; y 0–0.5, phase labels A/B/A
if R == 0:
    print("No runs; skip.")
else:
    for start_r, end_r in get_figure_chunks(R, 3):
        nrows_fig = end_r - start_r
        run_slice = run_labels[start_r:end_r]
        fig, axes = plt.subplots(nrows_fig, 3, figsize=(3 * SUBPLOT_W, nrows_fig * SUBPLOT_H), sharey=True, sharex=False)
        if nrows_fig == 1:
            axes = axes.reshape(1, -1)
        for r, run_label in enumerate(run_slice):
            for c, sched in enumerate(schedules):
                ax = axes[r, c]
                if run_label not in loss_results or sched not in loss_results[run_label]:
                    ax.set_visible(False)
                    continue
                loss = loss_results[run_label][sched]
                mean_loss = np.asarray(loss["mean"])
                steps = np.arange(len(mean_loss))
                bounds = loss.get("phase_boundaries")
                color = schedule_colours[c]
                ax.plot(steps, mean_loss, color=color, alpha=0.8, linewidth=1.5, label="winter (A2 untrained)" if (r, c) == (0, 0) else None)
                if loss_results_holton and run_label in loss_results_holton and sched in loss_results_holton[run_label]:
                    h = loss_results_holton[run_label][sched]
                    m = np.asarray(h["mean"])
                    n = min(len(m), len(steps))
                    ax.plot(steps[:n], m[:n], color=color, linestyle="--", alpha=0.8, linewidth=1.2, label="trained per phase (Holton-like)" if (r, c) == (0, 0) else None)
                if bounds is not None:
                    ax.axvline(bounds[0], color="k", linestyle="--", alpha=0.3)
                    ax.axvline(bounds[1], color="k", linestyle="--", alpha=0.3)
                    ax.set_xticks([0, bounds[0], bounds[1]])
                    ax.set_xticklabels(["A", "B", "A"])
                ax.set_ylim(0, 0.5)
                ax.set_yticks([0, 0.5])
                ax.set_xlabel("task")
                ax.set_title(display_label(run_label) + " | " + sched)
                if c == 0:
                    ax.set_ylabel(display_label(run_label), fontsize=7)
                else:
                    ax.set_ylabel("loss (MSE)")
                _style_axes(ax)
        if loss_results_holton:
            axes[0, 0].legend(fontsize=6)
        fig.suptitle("Training loss (Holton style). A2: winter = flat; trained per phase = re-learning.", y=1.02)
        plt.tight_layout()
        plt.show()

## 10. Holton loss only (trained per phase)

In [ ]:
# Same variant as steps comparison: trained per phase only (Holton-like); y 0–0.5, phase labels A/B/A
if R == 0:
    print("No runs; skip.")
else:
    for start_r, end_r in get_figure_chunks(R, 3):
        nrows_fig = end_r - start_r
        run_slice = run_labels[start_r:end_r]
        fig, axes = plt.subplots(nrows_fig, 3, figsize=(3 * SUBPLOT_W, nrows_fig * SUBPLOT_H), sharey=True, sharex=False)
        if nrows_fig == 1:
            axes = axes.reshape(1, -1)
        for r, run_label in enumerate(run_slice):
            for c, sched in enumerate(schedules):
                ax = axes[r, c]
                if run_label not in loss_results_holton or sched not in loss_results_holton[run_label]:
                    ax.set_visible(False)
                    continue
                loss = loss_results_holton[run_label][sched]
                mean_loss = np.asarray(loss["mean"])
                steps = np.arange(len(mean_loss))
                bounds = loss.get("phase_boundaries")
                color = schedule_colours[c]
                ax.plot(steps, mean_loss, color=color, alpha=0.8, linewidth=1.5)
                if bounds is not None:
                    ax.axvline(bounds[0], color="k", linestyle="--", alpha=0.3)
                    ax.axvline(bounds[1], color="k", linestyle="--", alpha=0.3)
                    ax.set_xticks([0, bounds[0], bounds[1]])
                    ax.set_xticklabels(["A", "B", "A"])
                ax.set_ylim(0, 0.5)
                ax.set_yticks([0, 0.5])
                ax.set_xlabel("task")
                ax.set_title(display_label(run_label) + " | " + sched)
                if c == 0:
                    ax.set_ylabel(display_label(run_label), fontsize=7)
                else:
                    ax.set_ylabel("loss (MSE)")
                _style_axes(ax)
        fig.suptitle("Training loss (trained per phase, Holton-like). A2 = summer re-learning.", y=1.02)
        plt.tight_layout()
        plt.show()

## 11. PCA n_components (99% variance)

Single plot: grouped bars by condition; one series per (run, task). Legend may be long when R is large.

In [ ]:
if R == 0:
    print("No runs; skip.")
else:
    pca_agg = pca_all.groupby(["run_label", "condition", "task"], as_index=False)["n_pca"].mean()
    x = np.arange(len(condition_order))
    w = 0.8 / max(2 * R, 1)
    fig, ax = plt.subplots(figsize=(6, max(3, R * 0.4)))
    for ri, run_label in enumerate(run_labels):
        for ti, task in enumerate(["post A", "post B"]):
            sub = pca_agg[(pca_agg["run_label"] == run_label) & (pca_agg["task"] == task)]
            means = [sub[(sub["condition"] == s)]["n_pca"].values for s in condition_order]
            means = [float(m[0]) if len(m) else 0 for m in means]
            off = (ri * 2 + ti) * w - 0.4
            ax.bar(x + off, means, w * 0.9, label="%s %s" % (display_label(run_label), task), alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(condition_order)
    ax.set_ylabel("n_pca")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=6)
    _style_axes(ax)
    plt.tight_layout()
    plt.show()

## 12. PCA 2D (post-A and post-B hiddens, hexagons)

One subplot per (run × condition). Uses `prepare_pca_single_task` and `plot_split_stim`. Split into multiple figures when R > MAX_ROWS_PER_FIGURE.

In [ ]:
schedule_to_idx = {"same": 0, "near": 1, "far": 2}

def get_hiddens_12(h, sched):
    if h is None or not hasattr(h, "shape"):
        return None
    if h.ndim == 3 and h.shape[0] == 3 and h.shape[1] >= 12:
        return h[schedule_to_idx[sched], :12, :]
    if h.ndim == 2 and len(h) >= 12:
        return h[:12]
    return None

SUBJ_IDX = 0
if R == 0:
    print("No runs; skip.")
else:
    for phase_key, phase_name in [("hiddens_post_phase_0", "post-A"), ("hiddens_post_phase_1", "post-B")]:
        for start_r, end_r in get_figure_chunks(R, 3):
            nrows_fig = end_r - start_r
            run_slice = run_labels[start_r:end_r]
            fig, axes = plt.subplots(nrows_fig, 3, figsize=(3 * SUBPLOT_W, nrows_fig * SUBPLOT_H), sharex=False, sharey=False)
            if nrows_fig == 1:
                axes = axes.reshape(1, -1)
            for r, run_label in enumerate(run_slice):
                if run_label not in all_ann_data:
                    continue
                data = all_ann_data[run_label]
                for c, sched in enumerate(schedules):
                    ax = axes[r, c]
                    if sched not in data or len(data[sched]) == 0:
                        ax.set_visible(False)
                        continue
                    subj = min(SUBJ_IDX, len(data[sched]) - 1)
                    h = data[sched][subj].get(phase_key)
                    h12 = get_hiddens_12(h, sched)
                    if h12 is None:
                        ax.set_visible(False)
                        continue
                    pca, transformed = ann.prepare_pca_single_task(h12)
                    lims = get_axis_limits(transformed)
                    plot_split_stim(ax, transformed, task_colours, lims)
                    ax.set_title(display_label(run_label) + " | " + sched)
                    if c == 0:
                        ax.set_ylabel(display_label(run_label), fontsize=7)
                    _style_axes(ax)
        plt.suptitle(phase_name + " hiddens (2D PCA)", y=1.02)
        plt.tight_layout()
        plt.show()